# Part A - Data preparation
# Loading both datasets and document

In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sentiment_df = pd.read_csv("fear_greed_index.csv")
trades_df = pd.read_csv("historical_data.csv")

In [122]:
print(sentiment_df.shape)
print(trades_df.shape)

(2644, 6)
(211224, 16)


In [123]:
print(sentiment_df.columns)
print(trades_df.columns)

Index(['timestamp', 'value', 'classification', 'date', 'Unnamed: 4',
       'Unnamed: 5'],
      dtype='object')
Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')


In [124]:
print(sentiment_df.head())

    timestamp  value classification        date  Unnamed: 4  Unnamed: 5
0  1517463000     30           Fear  01-02-2018         NaN         NaN
1  1517549400     15   Extreme Fear  02-02-2018         NaN         NaN
2  1517635800     40           Fear  03-02-2018         NaN         NaN
3  1517722200     24   Extreme Fear  04-02-2018         NaN         NaN
4  1517808600     11   Extreme Fear  05-02-2018         NaN         NaN


In [125]:
print(trades_df.head())

                                      Account  Coin  Execution Price  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9769   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9800   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9855   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9874   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed  @107           7.9894   

   Size Tokens  Size USD Side     Timestamp IST  Start Position Direction  \
0       986.87   7872.16  BUY  02-12-2024 22:50        0.000000       Buy   
1        16.00    127.68  BUY  02-12-2024 22:50      986.524596       Buy   
2       144.09   1150.63  BUY  02-12-2024 22:50     1002.518996       Buy   
3       142.98   1142.04  BUY  02-12-2024 22:50     1146.558564       Buy   
4         8.73     69.75  BUY  02-12-2024 22:50     1289.488521       Buy   

   Closed PnL                                   Transaction Hash     Order ID  \
0         0.0  0xec0945

In [126]:
print(sentiment_df.isnull().sum())
print(trades_df.isnull().sum())

timestamp            0
value                0
classification       0
date                 0
Unnamed: 4        2644
Unnamed: 5        2644
dtype: int64
Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64


In [127]:
print(sentiment_df.duplicated().sum())
print(trades_df.duplicated().sum())

0
0


In [128]:
print(sentiment_df.info())
print(trades_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   timestamp       2644 non-null   int64  
 1   value           2644 non-null   int64  
 2   classification  2644 non-null   object 
 3   date            2644 non-null   object 
 4   Unnamed: 4      0 non-null      float64
 5   Unnamed: 5      0 non-null      float64
dtypes: float64(2), int64(2), object(2)
memory usage: 124.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side         

# Key Observations
- Trader dataset contains 211,224 rows and 16 columns, indicating high-frequency trading activity data.
- Sentiment dataset contains 2,644 rows and 6 columns; includes 2 unnamed columns with no values, likely artifacts from CSV export (to be dropped).
- No missing values observed in meaningful columns across both datasets.
- No duplicate records found in either dataset.
- Trader dataset includes a timestamp column (likely in milliseconds), while sentiment data is at a daily level, indicating the need for time alignment and aggregation.
- Order IDs are repeated due to partial fills of a single order, while each Trade ID is unique, representing individual executions.

In [129]:
# Removes any columns whose names start with 'Unnamed'
# These are artifacts from CSV export and contain no useful values
sentiment_df = sentiment_df.loc[:,~sentiment_df.columns.str.contains('^Unnamed')]

# Convert timestamps and align the datasets by date

In [130]:
# Converts trade timestamp (stored in milliseconds) into datetime
trades_df['datetime'] = pd.to_datetime(trades_df['Timestamp'],unit = 'ms')

In [131]:
# Converts the datatype of Timestamp IST to datetime as it is in object
# Extracts only the date component from Timestamp IST not TimeStamp cause it's only giving 6rows after merging
trades_df.info()
trades_df['Timestamp IST'] = pd.to_datetime(trades_df['Timestamp IST'],dayfirst=True)
trades_df['date'] = pd.to_datetime(trades_df['Timestamp IST'].dt.date)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Account           211224 non-null  object        
 1   Coin              211224 non-null  object        
 2   Execution Price   211224 non-null  float64       
 3   Size Tokens       211224 non-null  float64       
 4   Size USD          211224 non-null  float64       
 5   Side              211224 non-null  object        
 6   Timestamp IST     211224 non-null  object        
 7   Start Position    211224 non-null  float64       
 8   Direction         211224 non-null  object        
 9   Closed PnL        211224 non-null  float64       
 10  Transaction Hash  211224 non-null  object        
 11  Order ID          211224 non-null  int64         
 12  Crossed           211224 non-null  bool          
 13  Fee               211224 non-null  float64       
 14  Trad

In [132]:
trades_df.sample(5)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,datetime,date
185448,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,HYPE,14.370,204.86,2943.84,SELL,2025-03-26 19:31:00,-433.430000,Open Short,0.000000,0xf3afc7ad34aea788f15f04204f8a280203940025e757...,81921235961,True,0.883151,9.150000e+14,1.740000e+12,2025-02-19 21:20:00,2025-03-26
178570,0xbee1707d6b44d4d52bfe19e41f8a828645437aab,@107,14.290,31.84,454.99,SELL,2025-03-11 12:43:00,1105.685295,Sell,0.292460,0x6495e665cbfb8f8dab01041f539bb70202b70024e0d9...,78974184013,True,0.136498,8.020000e+14,1.740000e+12,2025-02-19 21:20:00,2025-03-11
15029,0x083384f897ee0f19899168e3b1bec365f52a9012,SOL,277.000,3.19,883.63,SELL,2025-01-19 14:48:00,-4400.880000,Open Short,0.000000,0x033be037e7c034e4812b041bd49b5102035300f2d0c7...,64290912142,False,0.088363,1.570000e+14,1.740000e+12,2025-02-19 21:20:00,2025-01-19
117987,0x8170715b3b381dffb7062c0298972d4727a0a63b,TRUMP,8.600,22.80,196.08,SELL,2025-04-13 02:09:00,-39357.500000,Open Short,0.000000,0xb42e5110acedea4bf50504216c37fc02012400d7fe75...,85429453483,False,0.018823,2.250000e+14,1.740000e+12,2025-02-19 21:20:00,2025-04-13
132705,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,@4,19.899,3.00,59.70,SELL,2024-12-08 16:22:00,3.943140,Sell,43.729274,0xd9d3558149d82578de2b0418b8315a02016f00ee67e9...,53632839712,True,0.020893,5.950000e+14,1.730000e+12,2024-10-27 03:33:20,2024-12-08


In [133]:
# Convert trade timestamp (stored in milliseconds) into datetime
sentiment_df['datetime'] = pd.to_datetime(sentiment_df['timestamp'],unit = 's')
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'],dayfirst=True) #dayfirst = True to align with trades_df

In [134]:
sentiment_df.info()
sentiment_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       2644 non-null   int64         
 1   value           2644 non-null   int64         
 2   classification  2644 non-null   object        
 3   date            2644 non-null   datetime64[ns]
 4   datetime        2644 non-null   datetime64[ns]
dtypes: datetime64[ns](2), int64(2), object(1)
memory usage: 103.4+ KB


,timestamp,value,classification,date,datetime
0,1517463000,30,Fear,2018-02-01,2018-02-01 05:30:00
1,1517549400,15,Extreme Fear,2018-02-02,2018-02-02 05:30:00
2,1517635800,40,Fear,2018-02-03,2018-02-03 05:30:00
3,1517722200,24,Extreme Fear,2018-02-04,2018-02-04 05:30:00
4,1517808600,11,Extreme Fear,2018-02-05,2018-02-05 05:30:00


# Create the Key Metrics/Feature Engineering

In [135]:
account_daily = trades_df.groupby(['Account','date']).agg({
    'Closed PnL': 'sum',
    'Size USD': 'mean',
    'Order ID': 'count',
}).rename(columns={
    'Closed PnL':'daily PnL per trader',
    'Size USD':'Avg Trade Size',
    'Order ID':'Num of Trades'
}).reset_index()

# Overall Win Rate for each Account
trades_df['is_win'] = trades_df['Closed PnL'] > 0 #If Profit is > 0, it's a win else loss
win_rate = trades_df.groupby('Account')['is_win'].mean().reset_index()
win_rate.rename(columns={'is_win':'win rate'}, inplace = True)

# Merge win rate into account_daily
account_daily = account_daily.merge(
    win_rate,
    on = 'Account',
    how = 'left'
)

In [136]:
#Group by date,account,direction
long_short = trades_df.groupby(['date','Account','Side']).size().unstack(fill_value=0).reset_index()
     
#Compute Ratios
long_short['Long Ratio'] = long_short['BUY'] / (long_short['SELL'] + long_short['BUY'])
long_short['Short Ratio'] = long_short['SELL'] / (long_short['SELL'] + long_short['BUY'])

In [137]:
# Merge long/short ratios into account_daily
account_daily = account_daily.merge(
    long_short[['date','Account','Long Ratio','Short Ratio']],
    on = ['date','Account'],
    how = 'left'
)

In [138]:
print(account_daily.shape)
account_daily.sample(30).sort_values(by='date')

(2341, 8)


,Account,date,daily PnL per trader,Avg Trade Size,Num of Trades,win rate,Long Ratio,Short Ratio
321,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,2024-01-15,152.853580,69.984444,9,0.455215,0.000000,1.000000
1884,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2024-02-20,210.315294,427.575000,4,0.337134,0.000000,1.000000
1194,0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4,2024-02-27,312.130593,3759.745091,55,0.810876,0.600000,0.400000
711,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2024-04-13,0.000000,8677.150000,18,0.360364,1.000000,0.000000
748,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2024-06-11,0.000000,13143.021905,21,0.360364,1.000000,0.000000
773,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2024-07-17,0.000000,18326.035556,9,0.360364,0.000000,1.000000
782,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2024-07-26,0.000000,15000.965000,10,0.360364,1.000000,0.000000
805,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2024-08-28,0.000000,6784.937143,7,0.360364,1.000000,0.000000
1544,0x8477e447846c758f5a675856001ea72298fd9cb5,2024-11-11,4876.874304,1907.210407,172,0.261968,0.482558,0.517442
259,0x2c229d22b100a7beb69122eed721cee9b24011dd,2024-11-20,2626.800865,2627.910921,76,0.519914,0.342105,0.657895


In [139]:
merge_df = pd.merge(account_daily,
                    sentiment_df[['date','classification','value']],
                    on = 'date',how='inner')

In [140]:
print(merge_df.shape)
merge_df.sample(5)

(2340, 10)


,Account,date,daily PnL per trader,Avg Trade Size,Num of Trades,win rate,Long Ratio,Short Ratio,classification,value
1902,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2024-04-26,0.000000,1662.763333,3,0.337134,0.0,1.0,Greed,70
1080,0x6d6a4b953f202f8df5bed40692e7fd865318264a,2025-04-11,56.216535,2161.054000,5,0.431795,0.0,1.0,Fear,25
1758,0xa520ded057a32086c40e7dd6ed4eb8efb82c00e0,2024-08-18,0.000000,0.010000,1,0.573141,0.0,1.0,Fear,31
352,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,2025-03-10,-8224.491085,1653.866154,26,0.455215,0.0,1.0,Extreme Fear,20
239,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,2025-04-17,-15.688322,162.429000,10,0.438585,0.7,0.3,Fear,30


- Leverage distribution is a key risk metric, defined as Position Size ÷ Margin. In our dataset, we have notional trade sizes (Size USD) but lack margin/collateral fields, so exact leverage cannot be computed. As a proxy, we analyze trade size distribution to infer risk appetite. If margin data is added later, we can extend this analysis to true leverage distribution, plotting histograms and percentiles to capture trader behavior.